# Is the learned operator A still accurate on denoiser outputs?

In the route-2 sampler, the prox linearizes the frozen forward operator **A: CT1 → T1** at the
denoiser's estimate $\hat x = D(z_t, t) \approx \mathbb E[x_0 \mid z_t]$ -- a posterior MEAN, i.e. a
smoothed CT1, not a clean one. A was trained on clean CT1. This notebook measures whether that
matters, before anyone retrains A with blur augmentation.

For a fixed set of val slices and a grid of bridge positions $t$, it draws $z_t$ from the bridge,
runs a trained i2sb regressor, and compares (all inside the brain mask):

| quantity | meaning |
|---|---|
| `A_on_x0`   | RMSE of $T1 - A(x_0)$: A's own error on clean CT1 -- this is $\sigma_A$, the reference |
| `A_on_xhat` | RMSE of $T1 - A(\hat x)$: A's error where the prox actually evaluates it |
| `excess`    | `A_on_xhat - A_on_x0`: the part caused by feeding A a smoothed input |
| `den_err`   | RMSE of $\hat x - x_0$: how far the denoiser is from CT1 at this $t$ |
| `id_xhat`   | RMSE of $T1 - \hat x$: consistency with T1 if A were the identity |

each also split into the **enhancement proxy** (per slice, the top `ENH_Q` of CT1 − T1) and the rest.

**How to read it.** If `A_on_xhat` stays close to `A_on_x0` across $t$, the current A is fine as it
is. If it grows at large $t$ well beyond what `den_err` alone explains, A misbehaves on smooth
inputs and blur augmentation (or training on $\hat x$) is worth doing. Note that some excess is
expected and harmless: the prox's job is exactly to push $\hat x$ back towards consistency.

In [ ]:
import os, sys, json
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# NYUMets h5s store canonical-RAS axes, so a raw imshow puts the eyes on the image's RIGHT.
# DISPLAY ONLY -- the stored pixels and every checkpoint are untouched.
from visualization.image import set_display_orient
set_display_orient("radiological")           # "neurological", or None to switch it off

# ---- what to measure ---------------------------------------------------------------------------
# The DENOISER: a trained i2sb regressor's saved config (train.py writes <save_dir>/config.json at
# launch, and it points paths.ckpt at net.ckpt). Any regressor works. This one sees FLAIR/T1/T2
# (cond_idx [0, 1, 3]); the CT1-only A still gets no side information -- see the loader cell.
RUN_CONFIG = "trained_nets/nyumets/I2SB_Unet_NYUMets_CT1_from_all/config.json"

# The frozen OPERATOR A: a ladder checkpoint (scripts/fit_forward_ladder.py) or a train.py
# forward_op net.ckpt with its config.json beside it.
A_CKPT = "trained_nets/nyumets/forward_ladder_unet_cond_w8_CT1_to_T1/E_unet_w8_l3_x.pt"
A_COND_IDX = [3, 0]       # stored contrasts A takes as side information, IF it is conditioned
                          # (FLAIR, T1, CT1, T2 = 0..3). Ignored for a CT1-only A.
A_DATA = {"image_key": "img_median_mad", "scales": [3.0, 3.0, 3.0, 3.0]}   # what A was trained on

T_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]   # bridge positions (0 = CT1 end, 1 = T1 end)
N_SLICES = 400            # fixed random val subset
SLICE_RANGE = (40, 110)   # original slice indices [lo, hi) to use; overrides the run's config
                          # (older runs have none). None = the run's own setting
BATCH = 8
SEED = 0
ENH_Q = 0.98              # enhancement proxy: top 2% of CT1 - T1 per slice
PANEL_SLICE = 0           # which slice of the first batch to draw
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

## Load the denoiser, the schedule and A

The denoiser is rebuilt from its own saved config and checkpoint, and the schedule from that
config's `i2sb` block -- the one it was trained with. A's data scaling is not stored in its
checkpoint, so `A_DATA` states it and the next cell refuses a mismatch with the run's data: an
operator evaluated on the wrong intensity scale produces meaningless residuals, silently.

In [ ]:
import datasets                                    # noqa: F401  (registers loaders)
from datasets.registry import build_loader
from models import build_model
from training.common import load_ckpt
from training.i2sb import _split_batch
from training.forward_op import fixed_val_subset, enh_region
from sb.base import build_schedule, n_steps, forward_sample, forward_std, predict_x0
from sb.learned_dc import _load_E

with open(RUN_CONFIG) as f:
    cfg = yaml.safe_load(f)
if cfg.get("task") != "i2sb":
    raise ValueError(f"{RUN_CONFIG}: task is {cfg.get('task')!r}, not i2sb")
if cfg["i2sb"].get("learned_dc"):
    print("NOTE: this run trained WITH learned DC; here it is used as a plain denoiser (no DC).")

vcfg = dict(cfg["data"]["val"])
for k, v in A_DATA.items():
    if vcfg.get(k) != v:
        raise ValueError(f"A was trained with {k}={v}, but the run's val data has {k}={vcfg.get(k)}")

net = build_model(cfg).to(DEVICE).eval()
ckpt = cfg["paths"].get("ckpt") or os.path.join(cfg["paths"]["save_dir"], "net.ckpt")
load_ckpt(ckpt, model=net, device=DEVICE)

i2 = cfg["i2sb"]
sched = build_schedule(kind=i2.get("kind", "brownian"), tau=i2.get("tau", 0.19),
                       n_points=i2.get("n_points", 1000), beta_max=i2.get("beta_max", 0.3),
                       device=DEVICE)
if hasattr(net, "assert_schedule_matches"):
    net.assert_schedule_matches(sched)
target_channels = int(i2.get("target_channels", 1))

A, a_val_rmse = _load_E(A_CKPT, DEVICE)
print(f"denoiser: {type(net).__name__} from {ckpt}")
print(f"A: {A_CKPT}  cond_channels={A.cond_channels}  stored val rmse={a_val_rmse}")

## A fixed val subset

The loader is the run's own val loader, with `cond_idx` widened to include A's side information
when A is conditioned: the denoiser gets the channels it was trained on, A gets its own, both
sliced from one stack. The subset is fixed by `SEED`.

In [ ]:
run_cond = list(vcfg.get("cond_idx") or [])
a_cond = list(A_COND_IDX) if A.cond_channels else []
if A.cond_channels and len(a_cond) != A.cond_channels:
    raise ValueError(f"A takes {A.cond_channels} cond channel(s) but A_COND_IDX has {len(a_cond)}")
all_cond = run_cond + [c for c in a_cond if c not in run_cond]
net_sel = list(range(len(run_cond)))
a_sel = [all_cond.index(c) for c in a_cond]

vcfg.update(cond_idx=all_cond, batch_size=BATCH)
if SLICE_RANGE is not None:
    vcfg["slice_range"] = list(SLICE_RANGE)
full = build_loader(vcfg, shuffle=False, drop_last=False)
loader = fixed_val_subset(full, N_SLICES, SEED)
print(f"{len(loader.dataset)}/{len(full.dataset)} val slices | net cond {run_cond} | A cond {a_cond}")

## Measure

For every batch and every $t$: one bridge draw $z_t$ (seeded), one denoiser call, one A call.
$A(x_0)$ does not depend on $t$ and is computed once per batch. Everything is pooled over pixels
across the whole subset, not averaged per slice, so slices with more brain count more.

In [ ]:
REGIONS = ("all", "enh", "rest")
KEYS = ("A_on_xhat", "A_on_x0", "den_err", "id_xhat")
acc = {t: {k: {r: [0.0, 0.0] for r in REGIONS} for k in KEYS} for t in T_GRID}
panel = None
gen = torch.Generator(device="cpu").manual_seed(SEED)

def add(slot, err, masks):
    for r, m in masks.items():
        slot[r][0] += float((err ** 2 * m).sum())
        slot[r][1] += float(m.sum())

with torch.no_grad():
    for bi, batch in enumerate(loader):
        x0, x1, cond, mask, _, _ = _split_batch(batch, DEVICE)
        # T1 = the measurement A must reproduce. It is x1 for the T1 bridge; for a bridge that
        # starts at another study's CT1 the loader returns this session's T1 as "y".
        t1 = batch["y"].to(DEVICE) if isinstance(batch, dict) and "y" in batch else x1
        c_net = None if (cond is None or not net_sel) else cond[:, net_sel]
        c_A = None if not a_sel else cond[:, a_sel]
        m = (mask > 0.5).float()
        enh = enh_region(x0, t1, m, ENH_Q)
        masks = {"all": m, "enh": enh, "rest": m * (1 - enh)}
        a_x0 = A(x0, c_A)
        for t in T_GRID:
            step = torch.full((x0.shape[0],), int(round(t * (n_steps(sched) - 1))),
                              device=DEVICE, dtype=torch.long)
            torch.manual_seed(int(torch.randint(0, 2**31 - 1, (1,), generator=gen)))
            zt = forward_sample(sched, step, x0, x1)
            sigma = forward_std(sched, step, xdim=x0.shape[1:])
            xhat = predict_x0(net, zt, sigma, cond=c_net, target_channels=target_channels).real
            a_xhat = A(xhat, c_A)
            s = acc[t]
            add(s["A_on_xhat"], t1 - a_xhat, masks)
            add(s["A_on_x0"], t1 - a_x0, masks)
            add(s["den_err"], xhat - x0, masks)
            add(s["id_xhat"], t1 - xhat, masks)
            if bi == 0:
                panel = panel or {}
                i = min(PANEL_SLICE, x0.shape[0] - 1)
                panel[t] = dict(xhat=xhat[i, 0].cpu(), a_xhat=a_xhat[i, 0].cpu(),
                                x0=x0[i, 0].cpu(), x1=t1[i, 0].cpu(), a_x0=a_x0[i, 0].cpu(),
                                m=m[i, 0].cpu())

rmse = {t: {k: {r: (np.sqrt(v[0] / v[1]) if v[1] else np.nan) for r, v in acc[t][k].items()}
            for k in KEYS} for t in T_GRID}
print("done")

## Table

In [ ]:
hdr = f"{'t':>5s} | " + " | ".join(f"{k:>9s} {r:<4s}" for k in ("A_on_xhat", "A_on_x0") for r in REGIONS)
print(hdr + f" | {'excess':>7s} | {'den_err':>7s} | {'id_xhat':>7s}")
print("-" * len(hdr) + "-" * 32)
for t in T_GRID:
    R = rmse[t]
    row = " | ".join(f"{R[k][r]:14.4f}" for k in ("A_on_xhat", "A_on_x0") for r in REGIONS)
    exc = R["A_on_xhat"]["all"] - R["A_on_x0"]["all"]
    print(f"{t:5.2f} | {row} | {exc:7.4f} | {R['den_err']['all']:7.4f} | {R['id_xhat']['all']:7.4f}")
print(f"\nsigma_A stored in the checkpoint: {a_val_rmse}  (A_on_x0/all should be close to it)")

## Against $t$

Left: whole brain. Right: the enhancement proxy, where A has to do its real work and where a
smoothed $\hat x$ is most likely to confuse it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
styles = {"A_on_xhat": dict(marker="o", label=r"$T1 - A(\hat x)$"),
          "A_on_x0": dict(ls="--", label=r"$T1 - A(x_0)$  ($\sigma_A$)"),
          "den_err": dict(marker="s", alpha=0.7, label=r"$\hat x - x_0$  (denoiser)"),
          "id_xhat": dict(marker="^", alpha=0.7, label=r"$T1 - \hat x$  (A = identity)")}
for ax, region, title in zip(axes, ("all", "enh"), ("whole brain", "enhancement proxy")):
    for k, st in styles.items():
        ax.plot(T_GRID, [rmse[t][k][region] for t in T_GRID], **st)
    ax.set_title(title); ax.set_xlabel("bridge position t  (0 = CT1, 1 = T1)"); ax.set_ylabel("RMSE")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## One slice across $t$

Rows are bridge positions. Columns: the denoiser's $\hat x$, $A(\hat x)$, and the two residuals on
ONE fixed diverging window -- $T1 - A(\hat x)$ and, for reference, $T1 - A(x_0)$ (the same in every
row). Where the left residual lights up but the right one does not, the error comes from evaluating
A on a smoothed input rather than from A itself.

In [ ]:
from visualization.image import subplot_images

p0 = panel[T_GRID[0]]
ref = (p0["x1"] - p0["a_x0"])[p0["m"] > 0.5]
v = float(torch.quantile(ref.abs().float(), 0.99)) * 2 if ref.numel() else 1.0
rows, labels = [], []
for t in T_GRID:
    p = panel[t]
    rows.append([p["xhat"], p["a_xhat"], p["x1"] - p["a_xhat"], p["x1"] - p["a_x0"]])
    labels.append(f"t = {t:.2f}")
fig, _ = subplot_images(
    rows, row_labels=labels,
    col_titles=[r"x̂ (denoiser)", "A(x̂)", "T1 − A(x̂)", "T1 − A(x₀)"],
    cmap=["gray", "gray", "RdBu_r", "RdBu_r"],
    vmin=[None, None, -v, -v], vmax=[None, None, v, v],
    window_from=[p0["x0"], p0["x1"]], p=(1, 99), mask=p0["m"], apply_mask=True,
    magnitude=False, colorbar="each", panel_size=(2.9, 2.8), show=False)
plt.show()

## Decision

* **`A_on_xhat` ≈ `A_on_x0` everywhere**: keep the current A; the prox can use it as is.
* **Grows at large $t$, mostly in `enh`**: A misreads smoothed enhancement. Blur augmentation --
  CT1 blurred by a Gaussian whose width is drawn over the range of posterior-mean smoothing -- or
  training A directly on $\hat x$ is the fix. The $t$ where it departs is also a natural `t_max`
  for the prox.
* **Grows everywhere, tracking `den_err`**: that is mostly the denoiser's own error propagating
  through A (A ≈ identity outside enhancement), not A failing. That residual is what the prox is
  meant to correct.